# Nemotron Leaderboard Engine

Use this notebook as the Kaggle-side competition notebook for the NVIDIA Nemotron Model Reasoning Challenge.

Before running:
- Attach the competition input.
- Attach the Kaggle model `metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default`.
- Set `Settings -> Accelerator -> GPU RTX Pro 6000` first.
- Keep internet off unless the competition rules explicitly require it.

Run strategy:
1. Run the environment and input inspection cells.
2. Run the tokenizer-only cell.
3. Run the full model load cell.
4. If the model loads, run the short generation smoke test.
5. Only then build the real competition inference loop.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import torch

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Input root exists:', INPUT_ROOT.exists())
print('Working root:', WORK_ROOT)

if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f'GPU {index}: {torch.cuda.get_device_name(index)}')
        print('  total_memory_gb =', round(props.total_memory / 1024**3, 2))
else:
    print('No CUDA GPU detected. Stop here and turn on a GPU before loading the model.')

In [ ]:
top_level_inputs = sorted(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
print('Top-level /kaggle/input entries:')
for path in top_level_inputs:
    print('-', path)

sample_submission_candidates = sorted(INPUT_ROOT.rglob('sample_submission.csv'))
test_like_candidates = sorted(
    path for path in INPUT_ROOT.rglob('*')
    if path.name.lower() in {'test.csv', 'test.parquet', 'test.jsonl'}
)

print('\nSample submission candidates:')
for path in sample_submission_candidates[:10]:
    print('-', path)

print('\nTest file candidates:')
for path in test_like_candidates[:10]:
    print('-', path)

In [ ]:
def find_model_candidates(search_root: Path) -> list[Path]:
    candidates = []
    for config_path in search_root.rglob('config.json'):
        parent = config_path.parent
        if (parent / 'tokenizer_config.json').exists():
            candidates.append(parent)

    ordered = sorted(
        set(candidates),
        key=lambda path: ('nemotron' not in str(path).lower(), len(path.parts), str(path)),
    )
    return ordered


MODEL_CANDIDATES = find_model_candidates(INPUT_ROOT)
print('Model candidates:')
for index, path in enumerate(MODEL_CANDIDATES):
    print(index, path)

assert MODEL_CANDIDATES, 'No model root detected under /kaggle/input.'
MODEL_DIR = MODEL_CANDIDATES[0]
print('\nSelected MODEL_DIR =', MODEL_DIR)

SAMPLE_SUBMISSION_PATH = sample_submission_candidates[0] if sample_submission_candidates else None
TEST_DATA_PATH = test_like_candidates[0] if test_like_candidates else None
print('SAMPLE_SUBMISSION_PATH =', SAMPLE_SUBMISSION_PATH)
print('TEST_DATA_PATH =', TEST_DATA_PATH)

## Load The Tokenizer First

This should be cheap. If this fails, the attached model path is wrong or the Kaggle model input is incomplete.

Do not jump straight to the full model load when you are trying to preserve Kaggle hours.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
print('Tokenizer loaded from:', MODEL_DIR)
print('Tokenizer class:', tokenizer.__class__.__name__)
print('Tokenizer vocab size:', getattr(tokenizer, 'vocab_size', 'unknown'))

## Load The Model

This is the expensive step. If it fails with OOM on the selected Kaggle GPU, stop the session and do not keep retrying blindly.

The published NVIDIA example uses:
- `trust_remote_code=True`
- `torch_dtype=torch.bfloat16`
- `device_map='auto'`

In [ ]:
from transformers import AutoModelForCausalLM

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=dtype,
    trust_remote_code=True,
    device_map='auto',
)
model.eval()

def model_input_device(model):
    if hasattr(model, 'device'):
        return model.device
    return next(iter(model.parameters())).device


print('Model loaded successfully.')
print('Input device:', model_input_device(model))
print('Device map:', getattr(model, 'hf_device_map', 'single-device'))

In [ ]:
def tokenize_messages(messages, enable_thinking=None):
    kwargs = {
        'tokenize': True,
        'add_generation_prompt': True,
        'return_tensors': 'pt',
    }
    if enable_thinking is not None:
        try:
            return tokenizer.apply_chat_template(messages, enable_thinking=enable_thinking, **kwargs)
        except TypeError:
            print('Tokenizer does not expose enable_thinking; retrying without it.')
    return tokenizer.apply_chat_template(messages, **kwargs)


messages = [
    {
        'role': 'user',
        'content': 'Solve briefly: what is 17 * 19? Return only the final answer.',
    }
]

inputs = tokenize_messages(messages, enable_thinking=False).to(model_input_device(model))

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=32,
        do_sample=False,
        num_beams=1,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs.shape[-1]:]
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(generated_text)

In [ ]:
RUN_THINKING_TEST = False

if RUN_THINKING_TEST:
    thinking_inputs = tokenize_messages(messages, enable_thinking=True).to(model_input_device(model))
    with torch.no_grad():
        thinking_outputs = model.generate(
            thinking_inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=1.0,
            top_p=1.0,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = thinking_outputs[0][thinking_inputs.shape[-1]:]
    print(tokenizer.decode(generated_tokens, skip_special_tokens=False))
else:
    print('Set RUN_THINKING_TEST = True only after the short no-thinking smoke test succeeds.')

In [ ]:
if SAMPLE_SUBMISSION_PATH is not None:
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    print('Sample submission shape:', sample_submission.shape)
    print('Columns:', list(sample_submission.columns))
    display(sample_submission.head())

    prediction_columns = [
        column for column in sample_submission.columns
        if column.lower() not in {'id', 'index'}
    ]

    if prediction_columns:
        submission = sample_submission.copy()
        submission[prediction_columns[0]] = ''
        submission_path = WORK_ROOT / 'submission.csv'
        submission.to_csv(submission_path, index=False)
        print('Draft submission scaffold written to:', submission_path)
        display(submission.head())
    else:
        print('Could not infer the prediction column from the sample submission.')
else:
    print('sample_submission.csv was not found under /kaggle/input.')

## Next Step

If the model loads cleanly and the smoke test works on the selected GPU, replace the toy prompt cell with the real competition inference loop and build predictions from the competition test input.